<a href="https://colab.research.google.com/github/JWasonga/Statistical_Data_Analytics/blob/Applied_Statistics_and_Econometrics-_Portfolio/3_deeplearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('wordnet', quiet=True)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [3]:
from google.colab import files
uploaded = files.upload()

Saving reviews.txt to reviews.txt


In [5]:
df = pd.read_csv("reviews.txt")
df.head()

,date,reviewer_name,comments,cleaned_comments,polarity,sentiment
0,2016-08-20,Jenny,"Die Unterkunft war wirklicht toll, ruhig, nett...",die unterkunft war wirklicht toll ruhig nett z...,-0.7,negative
1,2023-01-08,Elia,"El apartamento estaba correcto, tal cual, como...",el apartamento estaba correcto tal cual como e...,-0.6,negative
2,2023-05-02,Cecilia,Yo buscaba estar cerca de la estación de Liver...,yo buscaba estar cerca de la estación de liver...,-0.6,negative
3,2018-11-25,Sibylle,"Elisas Wohnung liegt in Hampsted, mehrere Busl...",elisa wohnung liegt hampst mehrer buslinien ha...,-0.7,negative
4,2018-02-18,Nathalie,Ich habe meinen Aufenthalt bei Hedge und Britt...,ich habe meinen aufenthalt bei hedg und britt ...,-0.7,negative


#Date Preprocessing

In [6]:
df.isnull().sum()

,0
date,0
reviewer_name,0
comments,0
cleaned_comments,0
polarity,0
sentiment,0


In [8]:
#Dropping the Null values
df.dropna(inplace=True)
df.isnull().sum()

,0
date,0
reviewer_name,0
comments,0
cleaned_comments,0
polarity,0
sentiment,0


#Assessments:Experimentations (Using Deep Learning)

In [9]:
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word.isalpha()
              and word not in stop_words]

    return ' '.join(tokens)


In [11]:
#Applying pre-processing to cleaned comments
nltk.download('punkt_tab', quiet=True)
df['cleaned_comments'] = df['comments'].apply(preprocess_text)

In [12]:
#Tokenize Text
tokenizer = Tokenizer(num_words=500, oov_token="<OOV>")
tokenizer.fit_on_texts(df['cleaned_comments'])
sequences = tokenizer.texts_to_sequences(df['cleaned_comments'])


In [14]:
#Pad Sequences
max_length = max(len(x) for x in sequences)
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')

In [15]:
#Encode Labels
labels = pd.get_dummies(df['sentiment']).values

In [16]:
#Split the data
X_train, X_test, y_train, y_test = train_test_split(padded_sequences, labels, test_size=0.2, random_state=42)
#

In [17]:
model = Sequential()
model.add(Embedding(5000, 128, input_length=max_length))
model.add(LSTM(64, dropout=0.2, recurrent_dropout=0.2))
model.add(Dropout(0.5))
model.add(Dense(32, activation='relu'))
model.add(Dense(labels.shape[1], activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [18]:
early_stopping = EarlyStopping(monitor='val_loss', patience=3, verbose=1)
history = model.fit(X_train, y_train, epochs=100, batch_size=128, validation_split=0.2, callbacks=[early_stopping])

Epoch 1/100
58/58 ━━━━━━━━━━━━━━━━━━━━ 78s 1s/step - accuracy: 0.3256 - loss: 1.0997 - val_accuracy: 0.3279 - val_loss: 1.0990
Epoch 2/100
58/58 ━━━━━━━━━━━━━━━━━━━━ 74s 1s/step - accuracy: 0.3294 - loss: 1.0998 - val_accuracy: 0.3317 - val_loss: 1.0990
Epoch 3/100
58/58 ━━━━━━━━━━━━━━━━━━━━ 65s 1s/step - accuracy: 0.3328 - loss: 1.0994 - val_accuracy: 0.3279 - val_loss: 1.0993
Epoch 4/100
58/58 ━━━━━━━━━━━━━━━━━━━━ 62s 1s/step - accuracy: 0.3220 - loss: 1.0994 - val_accuracy: 0.3279 - val_loss: 1.0989
Epoch 5/100
58/58 ━━━━━━━━━━━━━━━━━━━━ 65s 1s/step - accuracy: 0.3296 - loss: 1.0991 - val_accuracy: 0.3279 - val_loss: 1.0987
Epoch 6/100
58/58 ━━━━━━━━━━━━━━━━━━━━ 78s 1s/step - accuracy: 0.3289 - loss: 1.0990 - val_accuracy: 0.3279 - val_loss: 1.0991
Epoch 7/100
58/58 ━━━━━━━━━━━━━━━━━━━━ 64s 1s/step - accuracy: 0.3294 - loss: 1.0991 - val_accuracy: 0.3279 - val_loss: 1.0992
Epoch 8/100
58/58 ━━━━━━━━━━━━━━━━━━━━ 64s 1s/step - accuracy: 0.3332 - loss: 1.0991 - val_accuracy: 0.3279 - v

In [19]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Test Accuracy: {accuracy * 100}%')

73/73 ━━━━━━━━━━━━━━━━━━━━ 7s 90ms/step - accuracy: 0.3395 - loss: 1.0986
Test Accuracy: 33.9516818523407%
